## Feature Engineering
New and meaningful features will be created from orders, customers and sellers tables

In [1]:
# Loading cleaned data from mysql
# copied code from the sql_analysis file
# connect to mysql
import pandas as pd
import os
import sqlalchemy as sqla
from dotenv import load_dotenv
from urllib.parse import quote_plus

# to load the .env file load_dotenv is used here
load_dotenv('../.env')

# all the credentials loaded from the .env file
username=os.getenv('DB_USER')
pwd=os.getenv('DB_PASSWORD')
host=os.getenv('DB_HOST')
port=os.getenv('DB_PORT')
db= os.getenv('DB_NAME')

# quote_plus encodes the password so that any special characters in password can't break url
safe_pwd=quote_plus(pwd)

# connection url
url='mysql+pymysql://'+username+':'+safe_pwd+'@'+host+':'+port

db_con=sqla.create_engine(url+'/'+db)

In [2]:
# loading data from sql
orders=pd.read_sql("select * from orders",db_con)
order_items=pd.read_sql("select * from order_items",db_con)
order_reviews=pd.read_sql("select * from order_reviews",db_con)
customers=pd.read_sql("select * from customers",db_con)

In [3]:
# checking shape to see if data is loaded properly
orders.shape, order_items.shape, order_reviews.shape,customers.shape

((99441, 8), (112650, 7), (99224, 7), (99441, 5))

In [4]:
# checking data types to see if data is loaded properly
orders.dtypes, order_items.dtypes, order_reviews.dtypes, customers.dtypes

(order_id                                 object
 customer_id                              object
 order_status                             object
 order_purchase_timestamp         datetime64[ns]
 order_approved_at                datetime64[ns]
 order_delivered_carrier_date     datetime64[ns]
 order_delivered_customer_date    datetime64[ns]
 order_estimated_delivery_date    datetime64[ns]
 dtype: object,
 order_id                       object
 order_item_id                   int64
 product_id                     object
 seller_id                      object
 shipping_limit_date    datetime64[ns]
 price                         float64
 freight_value                 float64
 dtype: object,
 review_id                          object
 order_id                           object
 review_score                        int64
 review_comment_title               object
 review_comment_message             object
 review_creation_date       datetime64[ns]
 review_answer_timestamp    datetime64[ns]
 d

#### Features creation

#### Orders features

In [5]:
# Total_value is order wise price+ freight_value
order_items['item_total']=order_items['price']+ order_items['freight_value']
order_value= order_items.groupby('order_id')['item_total'].sum().reset_index()
order_value=order_value.rename(columns={'item_total':'order_value'})
order_value.head()
# order_value.shape

,order_id,order_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


In [6]:
# delivery days and delivery delay features are created
orders['delivery_days']=(orders['order_delivered_customer_date']-orders['order_purchase_timestamp']).dt.days
orders['delivery_delay']=(orders['order_delivered_customer_date']-orders['order_estimated_delivery_date']).dt.days
orders[['delivery_days', 'delivery_delay']].describe()

# median around 10 for both the columns

,delivery_days,delivery_delay
count,96476.000000,96476.000000
mean,12.094086,-11.876881
std,9.551746,10.183854
min,0.000000,-147.000000
25%,6.000000,-17.000000
50%,10.000000,-12.000000
75%,15.000000,-7.000000
max,209.000000,188.000000


In [7]:
#merging all the created columns for orders table
order_features = orders[["order_id", "customer_id", "order_status", "order_purchase_timestamp","delivery_days", "delivery_delay"]]
order_features = order_features.merge(order_value, on="order_id", how="left")
order_features.shape

(99441, 7)

In [8]:
# adding customer_unique_id to for unique customer wise feature creation in next steps
cust_orders=order_features.merge(customers[['customer_id','customer_unique_id']], on='customer_id', how= 'left')
# keeping only cust_orders which have order_value
cust_orders=cust_orders[cust_orders['order_value'].notna()]
cust_orders.shape

(98666, 8)

#### Customers summary features

In [9]:
# creating table for customer_order_count and customer_total_spending
customer_summary=cust_orders.groupby('customer_unique_id').agg(order_count=('order_id','count'),
                                                               total_spending=('order_value','sum')).reset_index()
customer_summary['avg_order_value']=customer_summary['total_spending']/customer_summary['order_count']
customer_summary['is_repeat']=(customer_summary['order_count']>1).astype(int)
customer_summary.head()

# customer_summary table is created with the 4 new features 

,customer_unique_id,order_count,total_spending,avg_order_value,is_repeat
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,0
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,0


#### Sellers summary features

In [10]:
# sellers_summary table with 2 new features are created
seller_summary = order_items.groupby("seller_id").agg(seller_revenue=("price", "sum"),
                                                      seller_order_count=("order_id", "nunique")).reset_index()
seller_summary.head()

,seller_id,seller_revenue,seller_order_count
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00,3
1,001cca7ae9ae17fb1caed9dfb1094831,25080.03,200
2,001e6ad469a905060d959994f1b41e4f,250.00,1
3,002100f778ceb8431b7a1020ff7ab48f,1234.50,51
4,003554e2dce176b5555353e4f3555ac8,120.00,1


#### Loading the feature tables to sql

In [11]:
order_features.to_sql("order_features", db_con, if_exists="replace", index=False)
customer_summary.to_sql("customer_summary", db_con, if_exists="replace", index=False)
seller_summary.to_sql("seller_summary", db_con, if_exists="replace", index=False)

# these three feature tables are loaded to mysql

3095

In [12]:
# same code run after loaded tables to sql DB
tables = {'order_features': order_features,'customer_summary':customer_summary,'seller_summary': seller_summary}
for name in tables:
    # sql query to get the count of the table
    sql_count = pd.read_sql("select count(*) as cnt from " + name, db_con)['cnt'][0]
    print(name, len(tables[name]), sql_count)

order_features 99441 99441
customer_summary 95420 95420
seller_summary 3095 3095
